# Coortes ADNI — banda ±2 + selecção forward

Substitui o protocolo antigo de `1_dataset.ipynb` (só gap mínimo + extremos).

**Selecção:** `i1` = baseline; `i2`/`i3` = próximas imagens com gap na banda.

| Nominal | Banda |
|---------|-------|
| 6 m | `[4, 8]` (±2) |
| 12 m | `[10, 14]` (±2) |

±3 rejeitado: `[3,9]` e `[9,15]` partilham o 9. Ver `readme.md`.

Resultados antigos: `csvs/cohorts/{cohort}_old/`.

## 1) Carregar ADNI (sem repeats)

In [2]:
from pathlib import Path

import pandas as pd

INPUT_PATH = Path("csvs/adnimerged.csv")
DAYS_PER_MONTH = 30.4375

raw = pd.read_csv(INPUT_PATH)
raw["MRI_DATE"] = pd.to_datetime(raw["MRI_DATE"], errors="coerce")
is_repeat = raw["DESCRIPTION"].str.contains("repeat", case=False, na=False)
df = raw.loc[~is_repeat].dropna(subset=["MRI_DATE", "DIAG", "ID_PT"]).copy()
df = df.sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
_patients = list(df.groupby("ID_PT"))

print(f"Linhas originais: {len(raw)}")
print(f"Removidas (repeat): {int(is_repeat.sum())}")
print(f"Restantes: {len(df)}")
print(f"Pacientes: {df['ID_PT'].nunique()}")
print(f"Diagnósticos: {sorted(df['DIAG'].dropna().unique().tolist())}")

Linhas originais: 12921
Removidas (repeat): 3757
Restantes: 9164
Pacientes: 1303
Diagnósticos: ['AD', 'CN', 'MCI']


## 2) Tempo de conversão MCI → AD

Âncora = **primeira MRI MCI**. Conversão = **primeira MRI AD** depois dessa âncora.

In [3]:
rows = []
for id_pt, g in df.groupby("ID_PT"):
    g = g.drop_duplicates("MRI_DATE", keep="first")
    if not {"MCI", "AD"}.issubset(set(g["DIAG"])):
        continue
    first_mci = g.loc[g["DIAG"] == "MCI", "MRI_DATE"].min()
    first_ad = g.loc[g["DIAG"] == "AD", "MRI_DATE"].min()
    if pd.isna(first_mci) or pd.isna(first_ad) or first_ad <= first_mci:
        continue
    mci_before = g[(g["DIAG"] == "MCI") & (g["MRI_DATE"] < first_ad)]
    last_mci = mci_before["MRI_DATE"].max()
    rows.append(
        {
            "ID_PT": id_pt,
            "meses_1a_MCI→AD": (first_ad - first_mci).days / DAYS_PER_MONTH,
            "meses_última_MCI→AD": (first_ad - last_mci).days / DAYS_PER_MONTH,
            "n_MCI_antes_AD": len(mci_before),
        }
    )

conv = pd.DataFrame(rows)
print(f"Conversores MCI→AD: {len(conv)} pacientes\n")


def _stats(s: pd.Series) -> dict:
    return {
        "n": int(s.notna().sum()),
        "média": round(s.mean(), 2),
        "mediana": round(s.median(), 2),
        "dp": round(s.std(), 2),
        "Q1": round(s.quantile(0.25), 2),
        "Q3": round(s.quantile(0.75), 2),
        "mín": round(s.min(), 2),
        "máx": round(s.max(), 2),
    }


resumo_conv = pd.DataFrame(
    {
        "1ª MCI → 1ª AD": _stats(conv["meses_1a_MCI→AD"]),
        "última MCI → 1ª AD": _stats(conv["meses_última_MCI→AD"]),
    }
).T
print("Resumo (meses):")
display(resumo_conv)

janelas = [12, 18, 24, 36, 48, 60, 72]
cdf = []
for w in janelas:
    n = int((conv["meses_1a_MCI→AD"] <= w).sum())
    cdf.append(
        {
            "t_janela (meses)": w,
            "conversores capturados": n,
            "% dos conversores": round(100 * n / len(conv), 1),
        }
    )
print("\nCDF — % conversores com 1ª MCI→AD ≤ t_janela:")
display(pd.DataFrame(cdf))

Conversores MCI→AD: 218 pacientes

Resumo (meses):


,n,média,mediana,dp,Q1,Q3,mín,máx
1ª MCI → 1ª AD,218.0,25.23,19.5,17.33,12.47,36.22,5.49,97.58
última MCI → 1ª AD,218.0,10.39,6.7,7.39,5.98,12.42,2.79,61.31



CDF — % conversores com 1ª MCI→AD ≤ t_janela:


,t_janela (meses),conversores capturados,% dos conversores
0,12,43,19.7
1,18,89,40.8
2,24,117,53.7
3,36,161,73.9
4,48,180,82.6
5,60,208,95.4
6,72,214,98.2


## 3) Definição clínica + selecção forward (±2)

Altere `T_JANELA` e `T_IMAGENS` (centro nominal 6 ou 12). Banda = centro ± 2.

In [4]:
# === VARIÁVEIS (editar aqui) ===
T_JANELA = 36
T_IMAGENS = 6          # centro nominal; banda = ±2
QTD_IMAGENS = 3
SOFT_PMCI = True
GAP_TOL = 2.0          # ±2 meses (fixo)

GRADE_T_JANELA = [36, 48]
GRADE_T_IMAGENS = [6, 12]
# ===============================

DIAG_SEV = {"CN": 0, "MCI": 1, "AD": 2}
TARGET_GROUPS = ["CN", "sMCI", "pMCI", "AD"]


def band_limits(center: float, tol: float = GAP_TOL):
    """±2 fechado: 6→[4,8], 12→[10,14]. Evita partilha do 9 que ±3 teria."""
    lo, hi = center - tol, center + tol
    return lo, hi, True  # right_closed


def _dedup(g: pd.DataFrame) -> pd.DataFrame:
    return g.sort_values(["MRI_DATE", "ID_IMG"]).drop_duplicates("MRI_DATE", keep="first")


def _has_reversion(diags: list[str]) -> bool:
    sev = [DIAG_SEV[d] for d in diags if d in DIAG_SEV]
    return any(sev[i] > sev[i + 1] for i in range(len(sev) - 1))


def classify_patient(g: pd.DataFrame, t_janela: float):
    g = _dedup(g)
    if g.empty:
        return None, "sem_dados"

    first_diag = g.iloc[0]["DIAG"]
    t0 = g.iloc[0]["MRI_DATE"]
    t_end = t0 + pd.DateOffset(months=int(t_janela))

    if first_diag == "CN":
        if set(g["DIAG"]) != {"CN"}:
            return None, "trajetoria_mista"
        confirmation = g.loc[g["MRI_DATE"] >= t_end]
        if confirmation.empty:
            return None, "sem_confirmacao_pos_janela"
        return {
            "GROUP": "CN", "t0": t0, "t_end": t_end,
            "outcome_date": confirmation.iloc[0]["MRI_DATE"],
            "predictor_limit": t_end, "predictor_diag": "CN",
        }, "incluido"

    if first_diag == "AD":
        if set(g["DIAG"]) != {"AD"}:
            return None, "trajetoria_mista"
        return {
            "GROUP": "AD", "t0": t0, "t_end": t_end,
            "outcome_date": t0, "predictor_limit": t_end,
            "predictor_diag": "AD",
        }, "incluido"

    if first_diag != "MCI" or "CN" in set(g["DIAG"]):
        return None, "trajetoria_mista"

    if _has_reversion(g["DIAG"].tolist()):
        return None, "reversao_diagnostica"

    first_ad = g.loc[g["DIAG"] == "AD", "MRI_DATE"].min()
    if pd.notna(first_ad) and first_ad <= t_end:
        before_ad = g[(g["MRI_DATE"] >= t0) & (g["MRI_DATE"] < first_ad)]
        after_ad = g[g["MRI_DATE"] >= first_ad]
        if set(before_ad["DIAG"]) != {"MCI"} or set(after_ad["DIAG"]) != {"AD"}:
            return None, "trajetoria_mista"
        return {
            "GROUP": "pMCI", "t0": t0, "t_end": t_end,
            "outcome_date": first_ad, "predictor_limit": first_ad,
            "predictor_diag": "MCI",
        }, "incluido"

    confirmation = g.loc[g["MRI_DATE"] >= t_end]
    if confirmation.empty:
        return None, "sem_confirmacao_pos_janela"
    confirmation = confirmation.iloc[0]
    if confirmation["DIAG"] != "MCI":
        return None, "desfecho_intervalado"
    through_confirmation = g[(g["MRI_DATE"] >= t0) & (g["MRI_DATE"] <= confirmation["MRI_DATE"])]
    if set(through_confirmation["DIAG"]) != {"MCI"}:
        return None, "trajetoria_mista"

    return {
        "GROUP": "sMCI", "t0": t0, "t_end": t_end,
        "outcome_date": confirmation["MRI_DATE"],
        "predictor_limit": t_end, "predictor_diag": "MCI",
    }, "incluido"


def _gap_in_band(months: float, lo: float, hi: float, right_closed: bool) -> bool:
    if months < lo:
        return False
    return months <= hi if right_closed else months < hi


def _pick_forward_band(pool: pd.DataFrame, lo: float, hi: float, right_closed: bool):
    pool = _dedup(pool)
    if len(pool) < 3:
        return None
    dates = pool["MRI_DATE"].tolist()

    def next_in_band(i_from: int):
        for j in range(i_from + 1, len(dates)):
            gap = (dates[j] - dates[i_from]).days / DAYS_PER_MONTH
            if _gap_in_band(gap, lo, hi, right_closed):
                return j
        return None

    i2 = next_in_band(0)
    if i2 is None:
        return None
    i3 = next_in_band(i2)
    if i3 is None:
        return None
    return pool.iloc[[0, i2, i3]].copy()


def select_images(
    g: pd.DataFrame,
    clinical: dict,
    t_imagens: float,
    qtd: int,
    soft_pmci: bool,
):
    """Forward + banda ±GAP_TOL em torno de t_imagens. qtd deve ser 3."""
    if qtd != 3:
        raise ValueError("Protocolo forward±2 fixo em qtd=3.")

    g = _dedup(g)
    lo, hi, right_closed = band_limits(t_imagens)
    limit = clinical["predictor_limit"]
    before_limit = (
        g["MRI_DATE"] < limit
        if clinical["GROUP"] == "pMCI"
        else g["MRI_DATE"] <= limit
    )
    pool = g[
        (g["MRI_DATE"] >= clinical["t0"])
        & before_limit
        & (g["DIAG"] == clinical["predictor_diag"])
    ]
    selected = _pick_forward_band(pool, lo, hi, right_closed)
    used_soft = False

    if selected is None and soft_pmci and clinical["GROUP"] == "pMCI":
        first_ad = g[
            (g["MRI_DATE"] == clinical["outcome_date"]) & (g["DIAG"] == "AD")
        ]
        if not first_ad.empty:
            soft_pool = pd.concat([pool, first_ad.iloc[[0]]], ignore_index=True)
            selected = _pick_forward_band(soft_pool, lo, hi, right_closed)
            used_soft = selected is not None

    if selected is None:
        return None

    selected = selected.copy()
    selected["GROUP"] = clinical["GROUP"]
    selected["slot"] = [f"t{i}" for i in range(qtd)]
    selected["soft_pmci"] = used_soft
    selected["DIAG_EFFECTIVE"] = selected["DIAG"]
    if used_soft:
        if selected.iloc[-1]["DIAG"] != "AD" or (selected["DIAG"] == "AD").sum() != 1:
            raise ValueError("Soft pMCI deve conter exactamente 1 AD no último slot.")
        selected.loc[selected["DIAG"] == "AD", "DIAG_EFFECTIVE"] = "MCI_soft"

    selected["t0_anchor"] = clinical["t0"]
    selected["t_end"] = clinical["t_end"]
    selected["outcome_date"] = clinical["outcome_date"]

    if clinical["GROUP"] == "pMCI":
        mci_before_ad = g[
            (g["DIAG"] == "MCI")
            & (g["MRI_DATE"] >= clinical["t0"])
            & (g["MRI_DATE"] < clinical["outcome_date"])
        ]
        last_mci = mci_before_ad["MRI_DATE"].max()
        selected["last_mci_date"] = last_mci
        selected["last_mci_to_first_ad_months"] = (
            clinical["outcome_date"] - last_mci
        ).days / DAYS_PER_MONTH
    else:
        selected["last_mci_date"] = pd.NaT
        selected["last_mci_to_first_ad_months"] = float("nan")
    return selected


def classify_and_select(g, t_janela, t_imagens, qtd, soft_pmci):
    clinical, status = classify_patient(g, t_janela)
    if clinical is None:
        return None, None, status
    selected = select_images(g, clinical, t_imagens, qtd, soft_pmci)
    if selected is None:
        return clinical, None, "imagens_insuficientes"
    return clinical, selected, "incluido_soft" if selected["soft_pmci"].iloc[0] else "incluido"


def count_cohort(t_janela, t_imagens, qtd, soft_pmci=False, patients=None):
    clinical_counts = {group: 0 for group in TARGET_GROUPS}
    selected_counts = {group: 0 for group in TARGET_GROUPS}
    reasons = {}
    parts = []
    iterable = patients if patients is not None else df.groupby("ID_PT")

    for id_pt, g in iterable:
        clinical, selected, status = classify_and_select(
            g, t_janela, t_imagens, qtd, soft_pmci
        )
        reasons[status] = reasons.get(status, 0) + 1
        if clinical is None:
            continue
        clinical_counts[clinical["GROUP"]] += 1
        if selected is None:
            continue
        selected_counts[clinical["GROUP"]] += 1
        parts.append(selected)

    longitudinal = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    return clinical_counts, selected_counts, reasons, longitudinal


lo, hi, _ = band_limits(T_IMAGENS)
clinical_counts, counts, exclusion_reasons, longitudinal = count_cohort(
    T_JANELA, T_IMAGENS, QTD_IMAGENS, soft_pmci=SOFT_PMCI, patients=_patients
)

print("Configuração actual:")
display(
    pd.DataFrame(
        [
            {"parâmetro": "t_janela (meses)", "valor": T_JANELA},
            {"parâmetro": "t_imagens (centro)", "valor": T_IMAGENS},
            {"parâmetro": "banda", "valor": f"[{lo:g}, {hi:g}] (±{GAP_TOL:g})"},
            {"parâmetro": "qtd_imagens", "valor": QTD_IMAGENS},
            {"parâmetro": "SOFT_PMCI", "valor": SOFT_PMCI},
            {"parâmetro": "seleção", "valor": "forward a partir do baseline"},
        ]
    )
)

resumo = pd.DataFrame(
    [
        {
            "GROUP": group,
            "elegíveis clínicos": clinical_counts[group],
            "conjuntos com imagens": counts[group],
            "imagens": counts[group] * QTD_IMAGENS,
        }
        for group in TARGET_GROUPS
    ]
)
resumo.loc[len(resumo)] = {
    "GROUP": "All",
    "elegíveis clínicos": sum(clinical_counts.values()),
    "conjuntos com imagens": sum(counts.values()),
    "imagens": sum(counts.values()) * QTD_IMAGENS,
}
print("\nPopulação clínica e conjuntos finais (1 conjunto por paciente):")
display(resumo)
print(f"sMCI + pMCI finais = {counts['sMCI'] + counts['pMCI']}")
n_soft_current = (
    longitudinal.loc[longitudinal["soft_pmci"], "ID_PT"].nunique()
    if not longitudinal.empty else 0
)
print(f"pMCI soft = {n_soft_current}")

print("\nMotivos de exclusão / não selecção:")
display(
    pd.Series(exclusion_reasons, name="pacientes")
    .rename_axis("status")
    .sort_values(ascending=False)
    .reset_index()
)

if not longitudinal.empty and QTD_IMAGENS >= 2:
    slots = [f"t{i}" for i in range(QTD_IMAGENS)]
    wide = longitudinal.pivot_table(
        index=["ID_PT", "GROUP"], columns="slot", values="MRI_DATE", aggfunc="first"
    ).reindex(columns=slots)
    interval_rows = []
    groups_present = set(wide.index.get_level_values("GROUP"))
    for group in TARGET_GROUPS + ["All"]:
        if group != "All" and group not in groups_present:
            continue
        sub = wide if group == "All" else wide.xs(group, level="GROUP", drop_level=False)
        row = {"GROUP": group}
        for i in range(QTD_IMAGENS - 1):
            delta = (sub[slots[i + 1]] - sub[slots[i]]).dt.days / DAYS_PER_MONTH
            row[f"{slots[i]}→{slots[i + 1]}"] = f"{delta.mean():.2f} ± {delta.std():.2f}"
        span = (sub[slots[-1]] - sub[slots[0]]).dt.days / DAYS_PER_MONTH
        row[f"{slots[0]}→{slots[-1]}"] = f"{span.mean():.2f} ± {span.std():.2f}"
        interval_rows.append(row)
    print("\nIntervalos entre imagens (média ± dp, meses):")
    display(pd.DataFrame(interval_rows))

if GRADE_T_JANELA and GRADE_T_IMAGENS:
    grid_rows = []
    for tj in GRADE_T_JANELA:
        for ti in GRADE_T_IMAGENS:
            clinical, selected, _, grid_long = count_cohort(
                tj, ti, QTD_IMAGENS, soft_pmci=SOFT_PMCI, patients=_patients
            )
            n_soft = (
                grid_long.loc[grid_long["soft_pmci"], "ID_PT"].nunique()
                if not grid_long.empty else 0
            )
            blo, bhi, _ = band_limits(ti)
            grid_rows.append(
                {
                    "t_janela": tj,
                    "t_imagens": ti,
                    "banda": f"[{blo:g}, {bhi:g}]",
                    "CN": selected["CN"],
                    "sMCI": selected["sMCI"],
                    "pMCI": selected["pMCI"],
                    "pMCI soft": n_soft,
                    "AD": selected["AD"],
                    "total": sum(selected.values()),
                    "sMCI+pMCI": selected["sMCI"] + selected["pMCI"],
                }
            )
    grade = pd.DataFrame(grid_rows)
    print(f"\nGrade paper (±{GAP_TOL:g}; soft={SOFT_PMCI}):")
    display(grade)

Configuração actual:


,parâmetro,valor
0,t_janela (meses),36
1,t_imagens (centro),6
2,banda,"[4, 8] (±2)"
3,qtd_imagens,3
4,SOFT_PMCI,True
5,seleção,forward a partir do baseline



População clínica e conjuntos finais (1 conjunto por paciente):


,GROUP,elegíveis clínicos,conjuntos com imagens,imagens
0,CN,168,136,408
1,sMCI,168,125,375
2,pMCI,154,106,318
3,AD,269,149,447
4,All,759,516,1548


sMCI + pMCI finais = 231
pMCI soft = 46

Motivos de exclusão / não selecção:


,status,pacientes
0,incluido,470
1,sem_confirmacao_pos_janela,461
2,imagens_insuficientes,243
3,trajetoria_mista,48
4,incluido_soft,46
5,desfecho_intervalado,35



Intervalos entre imagens (média ± dp, meses):


,GROUP,t0→t1,t1→t2,t0→t2
0,CN,6.66 ± 0.52,6.06 ± 0.50,12.72 ± 0.61
1,sMCI,6.61 ± 0.62,6.05 ± 0.60,12.65 ± 0.84
2,pMCI,6.62 ± 0.65,6.01 ± 0.55,12.63 ± 0.69
3,AD,6.69 ± 0.45,6.01 ± 0.53,12.69 ± 0.58
4,All,6.65 ± 0.56,6.03 ± 0.54,12.68 ± 0.68



Grade paper (±2; soft=True):


,t_janela,t_imagens,banda,CN,sMCI,pMCI,pMCI soft,AD,total,sMCI+pMCI
0,36,6,"[4, 8]",136,125,106,46,149,516,231
1,36,12,"[10, 14]",131,121,33,33,86,371,154
2,48,6,"[4, 8]",100,73,120,46,149,442,193
3,48,12,"[10, 14]",99,72,48,33,86,305,120


## 4) Características clínicas — configuração manual

Define `T_JANELA_OPTIMO` / `T_IMAGENS_OPTIMO` e recalcula para tabela + testes sMCI×pMCI.

In [5]:
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu

T_JANELA_OPTIMO = 48
T_IMAGENS_OPTIMO = 12
QTD_IMAGENS_OPTIMO = 3
SOFT_PMCI_OPTIMO = True

clinical_counts_optimo, counts_optimo, exclusions_optimo, longitudinal_optimo = count_cohort(
    T_JANELA_OPTIMO,
    T_IMAGENS_OPTIMO,
    QTD_IMAGENS_OPTIMO,
    soft_pmci=SOFT_PMCI_OPTIMO,
    patients=_patients,
)

CLINICAL_VARS_OPTIMO = {
    "AGE": "Idade (anos)",
    "MMSE_SCORE": "MMSE",
    "ADAS_SCORE": "ADAS",
    "CDR_GLOBAL": "CDR global",
    "CDR_SB": "CDR-SB",
    "FAQ_SCORE": "FAQ",
}

if longitudinal_optimo.empty:
    raise ValueError("População vazia. Ajuste T_JANELA_OPTIMO / T_IMAGENS_OPTIMO.")

missing = [c for c in CLINICAL_VARS_OPTIMO if c not in longitudinal_optimo.columns]
if missing:
    raise ValueError(f"Colunas clínicas ausentes: {missing}")


def mean_std_n(series: pd.Series) -> str:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return ""
    if len(values) == 1:
        return f"{values.iloc[0]:.2f} ± — (n=1)"
    return f"{values.mean():.2f} ± {values.std():.2f} (n={len(values)})"


slots_optimo = [f"t{i}" for i in range(QTD_IMAGENS_OPTIMO)]
rows = []
for group in TARGET_GROUPS:
    for slot in slots_optimo:
        sub = longitudinal_optimo[
            (longitudinal_optimo["GROUP"] == group)
            & (longitudinal_optimo["slot"] == slot)
        ]
        if sub.empty:
            continue
        row = {
            "GROUP": group,
            "slot": slot,
            "pacientes": sub["ID_PT"].nunique(),
            "sexo M/F": f"{(sub['SEX'] == 'M').sum()}/{(sub['SEX'] == 'F').sum()}",
        }
        for column, label in CLINICAL_VARS_OPTIMO.items():
            row[label] = mean_std_n(sub[column])
        rows.append(row)

blo, bhi, _ = band_limits(T_IMAGENS_OPTIMO)
clinical_summary_optimo = pd.DataFrame(rows)
print(
    f"Configuração: t_janela={T_JANELA_OPTIMO}m | centro={T_IMAGENS_OPTIMO}m | "
    f"banda=[{blo:g}, {bhi:g}] | soft={SOFT_PMCI_OPTIMO}"
)
n_soft_optimo = longitudinal_optimo.loc[
    longitudinal_optimo["soft_pmci"], "ID_PT"
].nunique()
print(
    f"Conjuntos: {sum(counts_optimo.values())} | "
    f"Imagens: {len(longitudinal_optimo)} | "
    f"sMCI+pMCI: {counts_optimo['sMCI'] + counts_optimo['pMCI']} | "
    f"pMCI soft: {n_soft_optimo}"
)
display(clinical_summary_optimo)

STATS_SLOT = "t0"


def _fmt_p(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.4f}"


def _sig_label(p: float) -> str:
    if pd.isna(p):
        return ""
    return "sim (p<0.05)" if p < 0.05 else "não (p≥0.05)"


baseline_mci = (
    longitudinal_optimo[
        (longitudinal_optimo["GROUP"].isin(["sMCI", "pMCI"]))
        & (longitudinal_optimo["slot"] == STATS_SLOT)
    ]
    .drop_duplicates("ID_PT")
    .copy()
)
smci = baseline_mci[baseline_mci["GROUP"] == "sMCI"]
pmci = baseline_mci[baseline_mci["GROUP"] == "pMCI"]

stats_rows = []
sex_ct = pd.crosstab(baseline_mci["GROUP"], baseline_mci["SEX"]).reindex(
    index=["sMCI", "pMCI"], columns=["M", "F"], fill_value=0
)
if sex_ct.to_numpy().sum() > 0 and sex_ct.shape == (2, 2):
    expected = chi2_contingency(sex_ct)[3]
    if (expected < 5).any():
        _, p_sex = fisher_exact(sex_ct.to_numpy())
        sex_test = "Fisher exact"
    else:
        _, p_sex, _, _ = chi2_contingency(sex_ct)
        sex_test = "χ²"
else:
    p_sex = float("nan")
    sex_test = "—"

stats_rows.append(
    {
        "variável": "Sexo (M/F)",
        "slot": STATS_SLOT,
        "sMCI": f"{(smci['SEX'] == 'M').sum()}/{(smci['SEX'] == 'F').sum()} (n={len(smci)})",
        "pMCI": f"{(pmci['SEX'] == 'M').sum()}/{(pmci['SEX'] == 'F').sum()} (n={len(pmci)})",
        "teste": sex_test,
        "p-valor": _fmt_p(p_sex),
        "diferença significativa": _sig_label(p_sex),
    }
)

for column, label in CLINICAL_VARS_OPTIMO.items():
    a = pd.to_numeric(smci[column], errors="coerce").dropna()
    b = pd.to_numeric(pmci[column], errors="coerce").dropna()
    if len(a) == 0 or len(b) == 0:
        p_val = float("nan")
        test_name = "—"
    else:
        _, p_val = mannwhitneyu(a, b, alternative="two-sided")
        test_name = "Mann–Whitney U"
    stats_rows.append(
        {
            "variável": label,
            "slot": STATS_SLOT,
            "sMCI": mean_std_n(smci[column]),
            "pMCI": mean_std_n(pmci[column]),
            "teste": test_name,
            "p-valor": _fmt_p(p_val),
            "diferença significativa": _sig_label(p_val),
        }
    )

stats_rows.append(
    {
        "variável": "N pacientes (rótulo)",
        "slot": STATS_SLOT,
        "sMCI": str(len(smci)),
        "pMCI": str(len(pmci)),
        "teste": "razão sMCI:pMCI",
        "p-valor": f"{len(smci) / len(pmci):.2f}:1" if len(pmci) else "—",
        "diferença significativa": "—",
    }
)

print(f"\nsMCI × pMCI — testes no slot {STATS_SLOT}:")
display(pd.DataFrame(stats_rows))

audit_slots = [f"t{i}" for i in range(QTD_IMAGENS_OPTIMO)]
audit_groups = [g for g in ("CN", "sMCI", "pMCI", "AD") if g in set(longitudinal_optimo["GROUP"])]
if audit_groups and QTD_IMAGENS_OPTIMO >= 2:
    wide_audit = longitudinal_optimo.pivot_table(
        index=["ID_PT", "GROUP"], columns="slot", values="MRI_DATE", aggfunc="first"
    ).reindex(columns=audit_slots)
    interval_rows = []
    for group in audit_groups:
        sub = wide_audit.xs(group, level="GROUP", drop_level=False)
        row = {"GROUP": group, "pacientes": sub.shape[0]}
        for i in range(QTD_IMAGENS_OPTIMO - 1):
            delta = (sub[audit_slots[i + 1]] - sub[audit_slots[i]]).dt.days / DAYS_PER_MONTH
            row[f"{audit_slots[i]}→{audit_slots[i + 1]}"] = f"{delta.mean():.2f} ± {delta.std():.2f}"
        span = (sub[audit_slots[-1]] - sub[audit_slots[0]]).dt.days / DAYS_PER_MONTH
        row[f"{audit_slots[0]}→{audit_slots[-1]}"] = f"{span.mean():.2f} ± {span.std():.2f}"
        interval_rows.append(row)
    print("\nAuditoria temporal (meses):")
    display(pd.DataFrame(interval_rows))

Configuração: t_janela=48m | centro=12m | banda=[10, 14] | soft=True
Conjuntos: 305 | Imagens: 915 | sMCI+pMCI: 120 | pMCI soft: 33


,GROUP,slot,pacientes,sexo M/F,Idade (anos),MMSE,ADAS,CDR global,CDR-SB,FAQ
0,CN,t0,99,47/52,75.52 ± 6.29 (n=99),29.20 ± 1.10 (n=99),5.42 ± 2.63 (n=99),0.00 ± 0.00 (n=99),0.02 ± 0.09 (n=99),0.08 ± 0.27 (n=99)
1,CN,t1,99,47/52,76.58 ± 6.27 (n=99),28.98 ± 1.33 (n=99),5.22 ± 2.80 (n=99),0.02 ± 0.09 (n=99),0.03 ± 0.12 (n=99),0.16 ± 1.05 (n=99)
2,CN,t2,99,47/52,77.59 ± 6.31 (n=99),29.25 ± 0.98 (n=99),5.22 ± 2.79 (n=99),0.01 ± 0.07 (n=99),0.04 ± 0.22 (n=99),0.12 ± 0.64 (n=99)
3,sMCI,t0,72,49/23,72.88 ± 6.70 (n=72),28.00 ± 1.63 (n=72),8.71 ± 3.40 (n=72),0.50 ± 0.00 (n=72),1.07 ± 0.59 (n=72),1.85 ± 3.30 (n=72)
4,sMCI,t1,72,49/23,73.93 ± 6.73 (n=72),28.32 ± 1.69 (n=72),8.07 ± 3.65 (n=72),0.45 ± 0.17 (n=72),1.17 ± 0.89 (n=72),2.43 ± 3.37 (n=72)
5,sMCI,t2,72,49/23,74.96 ± 6.68 (n=72),28.29 ± 1.86 (n=72),8.65 ± 3.59 (n=72),0.44 ± 0.18 (n=72),1.29 ± 1.03 (n=72),2.26 ± 3.26 (n=72)
6,pMCI,t0,48,28/20,73.98 ± 6.83 (n=48),26.94 ± 1.56 (n=48),12.18 ± 2.95 (n=48),0.50 ± 0.00 (n=48),1.93 ± 1.10 (n=48),4.83 ± 4.88 (n=48)
7,pMCI,t1,48,28/20,75.00 ± 6.83 (n=48),25.90 ± 1.95 (n=48),13.24 ± 3.79 (n=48),0.52 ± 0.10 (n=48),2.45 ± 1.18 (n=48),6.77 ± 5.36 (n=48)
8,pMCI,t2,48,28/20,76.06 ± 6.86 (n=48),23.90 ± 2.81 (n=48),16.44 ± 4.81 (n=48),0.71 ± 0.31 (n=48),3.76 ± 1.61 (n=48),10.56 ± 6.51 (n=48)
9,AD,t0,86,46/40,75.51 ± 7.64 (n=86),22.90 ± 1.85 (n=86),18.91 ± 5.72 (n=86),0.78 ± 0.28 (n=86),4.40 ± 1.54 (n=86),12.81 ± 6.93 (n=86)



sMCI × pMCI — testes no slot t0:


,variável,slot,sMCI,pMCI,teste,p-valor,diferença significativa
0,Sexo (M/F),t0,49/23 (n=72),28/20 (n=48),χ²,0.3714,não (p≥0.05)
1,Idade (anos),t0,72.88 ± 6.70 (n=72),73.98 ± 6.83 (n=48),Mann–Whitney U,0.5196,não (p≥0.05)
2,MMSE,t0,28.00 ± 1.63 (n=72),26.94 ± 1.56 (n=48),Mann–Whitney U,2.21e-04,sim (p<0.05)
3,ADAS,t0,8.71 ± 3.40 (n=72),12.18 ± 2.95 (n=48),Mann–Whitney U,4.08e-07,sim (p<0.05)
4,CDR global,t0,0.50 ± 0.00 (n=72),0.50 ± 0.00 (n=48),Mann–Whitney U,1.0000,não (p≥0.05)
5,CDR-SB,t0,1.07 ± 0.59 (n=72),1.93 ± 1.10 (n=48),Mann–Whitney U,5.35e-06,sim (p<0.05)
6,FAQ,t0,1.85 ± 3.30 (n=72),4.83 ± 4.88 (n=48),Mann–Whitney U,5.49e-05,sim (p<0.05)
7,N pacientes (rótulo),t0,72,48,razão sMCI:pMCI,1.50:1,—



Auditoria temporal (meses):


,GROUP,pacientes,t0→t1,t1→t2,t0→t2
0,CN,99,12.67 ± 0.50,12.20 ± 0.61,24.86 ± 0.57
1,sMCI,72,12.75 ± 0.52,12.15 ± 0.65,24.90 ± 0.71
2,pMCI,48,12.52 ± 0.61,12.15 ± 0.64,24.67 ± 0.77
3,AD,86,12.64 ± 0.42,12.14 ± 0.56,24.79 ± 0.62


## 5) Gravar coortes paper (anti-overwrite)

Gera `36m_6m`, `36m_12m`, `48m_6m`, `48m_12m` + `all_population`.
Falha se a pasta nova já existir (legado está em `*_old`).

In [9]:
COHORTS_DIR = Path("csvs/cohorts")
PAPER_CONFIGS = [(36, 6), (36, 12), (48, 6), (48, 12)]
SELECTION_TAG = f"forward_band_pm{int(GAP_TOL)}"


def save_cohort(t_janela: int, t_imagens: int, soft_pmci: bool = True) -> Path:
    cohort = f"{t_janela}m_{t_imagens}m"
    cohort_dir = COHORTS_DIR / cohort
    output = cohort_dir / "adnimerged_longitudinal.csv"
    if output.exists():
        raise FileExistsError(
            f"Já existe {output}. Apague ou use pasta *_old."
        )

    _, counts, _, long = count_cohort(
        t_janela, t_imagens, QTD_IMAGENS, soft_pmci=soft_pmci, patients=_patients
    )
    if long.empty:
        raise ValueError(f"População vazia: {cohort}")

    sizes = long.groupby("ID_PT").size()
    if not sizes.eq(QTD_IMAGENS).all():
        raise ValueError(f"{cohort}: cada paciente deve ter exactamente {QTD_IMAGENS} linhas.")
    if long.groupby("ID_PT")["GROUP"].nunique().gt(1).any():
        raise ValueError(f"{cohort}: paciente com mais de um GROUP.")

    lo, hi, _ = band_limits(t_imagens)
    # self-check gaps na banda (sMCI/pMCI e restantes)
    wide = long.pivot_table(
        index="ID_PT", columns="slot", values="MRI_DATE", aggfunc="first"
    )
    g12 = (wide["t1"] - wide["t0"]).dt.days / DAYS_PER_MONTH
    g23 = (wide["t2"] - wide["t1"]).dt.days / DAYS_PER_MONTH
    assert g12.between(lo, hi).all() and g23.between(lo, hi).all(), f"gap fora de [{lo},{hi}]"

    slot_order = {f"t{i}": i for i in range(QTD_IMAGENS)}
    out = long.copy()
    out["PARAM_T_JANELA"] = t_janela
    out["PARAM_T_IMAGENS"] = t_imagens
    out["PARAM_GAP_LO"] = lo
    out["PARAM_GAP_HI"] = hi
    out["PARAM_GAP_TOL"] = GAP_TOL
    out["PARAM_QTD_IMAGENS"] = QTD_IMAGENS
    out["PARAM_SOFT_PMCI"] = soft_pmci
    out["PARAM_SELECTION"] = SELECTION_TAG
    out["_slot_order"] = out["slot"].map(slot_order)
    out = (
        out.sort_values(["GROUP", "ID_PT", "_slot_order", "MRI_DATE", "ID_IMG"])
        .drop(columns="_slot_order")
    )

    cohort_dir.mkdir(parents=True, exist_ok=True)
    out.to_csv(output, index=False)
    print(
        f"Salvo {output} | sMCI={counts['sMCI']} pMCI={counts['pMCI']} "
        f"CN={counts['CN']} AD={counts['AD']} | banda=[{lo:g},{hi:g}]"
    )
    return output


saved = []
for tj, ti in PAPER_CONFIGS:
    saved.append(save_cohort(tj, ti, soft_pmci=SOFT_PMCI))

# União para extracção de atributos
OUT_DIR = COHORTS_DIR / "all_population"
OUT_CSV = OUT_DIR / "all_population.csv"
if OUT_CSV.exists():
    raise FileExistsError(f"Já existe {OUT_CSV}")

REQUIRED = ["ID_PT", "ID_IMG", "SEX", "AGE", "MRI_DATE"]
KEEP_EXTRA = ["GROUP", "slot", "DIAG", "DIAG_EFFECTIVE", "soft_pmci"]
parts = []
for path in saved:
    d = path.parent
    part = pd.read_csv(path)
    missing = set(REQUIRED) - set(part.columns)
    if missing:
        raise ValueError(f"{d.name}: faltam {sorted(missing)}")
    part = part.copy()
    part["ID_IMG"] = part["ID_IMG"].astype(str).str.strip()
    part["ID_PT"] = part["ID_PT"].astype(str).str.strip()
    part["MRI_DATE"] = pd.to_datetime(part["MRI_DATE"], errors="coerce")
    part["_source_cohort"] = d.name
    parts.append(part)

raw_u = pd.concat(parts, ignore_index=True)
cols = REQUIRED + [c for c in KEEP_EXTRA if c in raw_u.columns] + ["_source_cohort"]
union = (
    raw_u[cols]
    .sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
    .drop_duplicates("ID_IMG", keep="first")
    .reset_index(drop=True)
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
union.to_csv(OUT_CSV, index=False)
print(f"\nSalvo {OUT_CSV}")
print(f"Imagens: {len(union)} | Pacientes: {union['ID_PT'].nunique()}")
print(raw_u.groupby("_source_cohort")["ID_IMG"].nunique().rename("n_img").to_string())
print("OK — coortes prontas para 4_run_post_extract / 5_*")

FileExistsError: Já existe csvs/cohorts/36m_6m/adnimerged_longitudinal.csv. Apague ou use pasta *_old.